In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Define dataset parameters
data_dir = "../data/flower_photos/"  # Path to your extracted folder
batch_size = 32
img_height = 180
img_width = 180

# 2. Load Training Dataset (80% split)
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# 3. Load Validation Dataset (20% split)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Detected Flower Classes:", class_names)

# Optimize dataset loading performance for CPU
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 4. Build Model using Transfer Learning (MobileNetV2 is lightweight and fast on CPU)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base layers

model = models.Sequential([
    layers.Rescaling(1./127.5, offset=-1, input_shape=(img_height, img_width, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

# 5. Compile Model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 6. Train the Model
epochs = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

print("Training finished!")

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Detected Flower Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


C:\Users\PERSONAL\AppData\Local\Temp\ipykernel_18308\996343267.py:39: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 6, 6, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,264,389 (8.64 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 44s 404ms/step - accuracy: 0.6812 - loss: 0.8765 - val_accuracy: 0.8406 - val_loss: 0.4470
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 36s 393ms/step - accuracy: 0.8508 - loss: 0.4353 - val_accuracy: 0.8787 - val_loss: 0.3627
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 32s 352ms/step - accuracy: 0.8665 - loss: 0.3642 - val_accuracy: 0.8869 - val_loss: 0.3368
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 34s 373ms/step - accuracy: 0.8903 - loss: 0.3009 - val_accuracy: 0.8787 - val_loss: 0.3248
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 39s 426ms/step - accuracy: 0.9070 - loss: 0.2704 - val_accuracy: 0.8774 - val_loss: 0.3298
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 35s 386ms/step - accuracy: 0.9169 - loss: 0.2368 - val_accuracy: 0.8842 - val_loss: 0.3131
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 35s 385ms/step - accuracy: 0.9285 - loss: 0.2183 - val_accuracy: 0.8815 - val_loss: 0.3177
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 34s 370ms/step - accuracy: 0.9240 - loss: 0.2110 - val_accu

In [9]:
import tensorflow as tf
import numpy as np

class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']

img_height = 180
img_width = 180

image_path = "../data/tulip2.jpg"  # Replace with your actual image path
img = tf.keras.utils.load_img(
    image_path, target_size=(img_height, img_width)
)

img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

predictions = model.predict(img_array)

score = tf.nn.softmax(predictions[0])

predicted_class = class_names[np.argmax(score)]
confidence = 100 * np.max(score)

print(f"Predicted Flower: {predicted_class}")
print(f"Confidence Score: {confidence:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Predicted Flower: tulips
Confidence Score: 38.86%
